# randn-like-noise-source — ex1: reparameterization noise that inherits dtype

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `randn-like-noise-source`. Running the final beacon cell reports progress against the `Generative: randn-like noise source` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: randn-like noise source` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`randn-like-noise-source`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "randn-like-noise-source"
DD_SUBTOPIC = "Generative: randn-like noise source"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `randn_like` noise source — quick refresher

`t.randn_like(x)` returns a standard-normal tensor with the SAME shape, dtype, and device as `x`. It is the canonical noise source for VAE reparameterization (`z = mu + sigma * t.randn_like(sigma)`).

**Compared to `t.randn(*x.shape)`.** That call also matches shape, but produces a `float32` tensor on the CPU — silently breaks when `x` is on GPU or half-precision. Symptoms: `RuntimeError: expected all tensors to be on the same device` or huge numerical drift from a float32-on-fp16 graph.

**Rule of thumb.** Any time the noise needs to be added to / multiplied by an existing tensor `x`, use `randn_like(x)` — it inherits all three of (shape, dtype, device) so the downstream op is always well-typed.

### Exercise 1 — reparameterization noise that inherits dtype

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `t.randn_like(sigma)` (NOT `t.randn(*sigma.shape)`) to sample VAE reparameterization noise that automatically inherits the target's dtype and device.
> Keywords: randn_like, vae, reparameterization, dtype
> ```

**KCs targeted:** `randn-like-preserves-dtype`, `randn-like-preserves-shape`

Implement `ex1_reparameterize(mu, sigma)`. The VAE reparameterization trick — the bridge that lets gradients flow through a stochastic sampling step:

1. `mu` and `sigma` both have shape `(B, latent_dim)` and matching dtype (possibly `float64` or `float16`, not just `float32`).
2. Sample `eps` from a standard normal using `t.randn_like(sigma)` — the SHAPE, DTYPE, AND DEVICE all inherit from `sigma`. **Do NOT** use `t.randn(*sigma.shape)` (that hardcodes `float32` / CPU).
3. Compute `z = mu + sigma * eps`.
4. Return `z`.

Input: `mu`, `sigma` — `(B, latent_dim)` float tensors (any float dtype).
Output: `z` — `(B, latent_dim)` float tensor, same dtype as `mu` / `sigma`.

The visualization plots ε samples from `randn_like` against ε samples from `randn(*shape)` to show that both produce standard-normal noise — but only `randn_like` keeps the dtype contract.

In [ ]:
def ex1_reparameterize(mu: Tensor, sigma: Tensor) -> Tensor:
    eps = t.randn_like(sigma)
    return mu + sigma * eps


<details><summary>Solution</summary>

```python
def ex1_reparameterize(mu: Tensor, sigma: Tensor) -> Tensor:
    eps = t.randn_like(sigma)
    return mu + sigma * eps
```

**`randn_like(sigma)` vs `randn(*sigma.shape)`.** Both produce standard-normal noise of the right shape. ONLY `randn_like` preserves dtype and device. The shape-call returns CPU `float32` no matter what — fine until you move the model to GPU or switch to mixed precision, at which point you get a `RuntimeError: expected all tensors to be on the same device` or silent numerical drift.

**Why the noise is multiplied by sigma, not just added.** This is the heart of the reparameterization trick. `z ~ N(mu, sigma^2)` is equivalent in distribution to `mu + sigma * eps` with `eps ~ N(0, 1)`. The latter form is differentiable through `mu` and `sigma` (the stochasticity is in `eps`, which doesn't carry gradients), so the encoder learns from the decoder.

**`sigma`, not `log_sigma`, in this drill.** ARENA actually has the encoder output `log_sigma` and computes `sigma = log_sigma.exp()` before this step (so sigma stays positive). We're drilling the noise-injection layer in isolation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()